# Lab 2 Notebook 2: From Notebook to REST API

This notebook shows students how notebook logic becomes a real FastAPI web app.

## Step 1: Load the trained model artifact from Notebook 1

In [ ]:
from pathlib import Path
import pickle
import numpy as np

model_path = Path("..") / "model.pkl"
if not model_path.exists():
    raise FileNotFoundError("Run model.ipynb first to generate ../model.pkl")

with model_path.open("rb") as f:
    model = pickle.load(f)

SPECIES = ["setosa", "versicolor", "virginica"]

def predict_species(sepal_length, sepal_width, petal_length, petal_width):
    features = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
    class_index = int(model.predict(features)[0])
    return {
        "prediction": SPECIES[class_index],
        "class_index": class_index,
    }

## Step 2: Quick notebook prediction check

In [ ]:
predict_species(5.1, 3.5, 1.4, 0.2)

## Step 3: Generate the production API file (`../main.py`)

Students can see the exact code that moves from notebook experimentation to an actual web app endpoint.

In [ ]:
main_py = """from pathlib import Path
import pickle

import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(
    title="Iris Classifier API",
    description="Predict Iris flower species from measurements",
    version="1.0.0",
)

model_path = Path(__file__).resolve().parent / "model.pkl"
with model_path.open("rb") as f:
    model = pickle.load(f)

SPECIES = ["setosa", "versicolor", "virginica"]


class IrisInput(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

    class Config:
        json_schema_extra = {
            "example": {
                "sepal_length": 5.1,
                "sepal_width": 3.5,
                "petal_length": 1.4,
                "petal_width": 0.2,
            }
        }


class PredictionOutput(BaseModel):
    prediction: str
    class_index: int


@app.get("/")
def root():
    return {"message": "Iris Classifier API is running. Visit /docs to test."}


@app.post("/predict", response_model=PredictionOutput)
def predict(data: IrisInput):
    features = np.array(
        [[data.sepal_length, data.sepal_width, data.petal_length, data.petal_width]]
    )
    class_index = int(model.predict(features)[0])
    return PredictionOutput(prediction=SPECIES[class_index], class_index=class_index)
"""

output_path = Path("..") / "main.py"
output_path.write_text(main_py)
print(f"Generated {output_path.resolve()}")

## Step 4: Run the web app (terminal, from `lab2-ml-api`)
```bash
uvicorn main:app --reload
```
Then open `http://localhost:8000/docs` and test `POST /predict`.